# AI vs. Human Text Classifier
**Binary Text Classification — DAIGT V2 Dataset**

| | |
|---|---|
| Task | Classify essays as Human-written (0) or AI-generated (1) |
| Model | Logistic Regression + TF-IDF (unigrams + bigrams) |
| Target Accuracy | 86 – 90% |
| Dataset | [DAIGT V2 Train Dataset](https://www.kaggle.com/datasets/thedrcat/daigt-v2-train-dataset) |

## Cell 1 — Install & Import Dependencies

In [2]:
# Install required libraries (works in local Jupyter and Colab)
%pip install nltk scikit-learn pandas numpy matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import pickle
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Download required NLTK assets
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

print('All libraries loaded successfully.')

Note: you may need to restart the kernel to use updated packages.
All libraries loaded successfully.


## Cell 2 — Load Dataset

Download `train_v2_drcat_02.csv` from [Kaggle](https://www.kaggle.com/datasets/thedrcat/daigt-v2-train-dataset) first.

ilePlace the CSV in the same folder as this notebook, or update `DATA_PATH` in the next cell to point to the file on your laptop.

In [ ]:
# Load the dataset from a local file path.
# Place train_v2_drcat_02.csv in the same folder as this notebook,
# or set DATA_PATH to the full path on your laptop.
from pathlib import Path

DATA_PATH = Path('train_v2_drcat_02.csv')

if not DATA_PATH.exists():
    candidates = list(Path.cwd().glob('**/train_v2_drcat_02.csv'))
    if candidates:
        DATA_PATH = candidates[0]
        print(f'Found dataset at: {DATA_PATH}')
    else:
        raise FileNotFoundError(
            'Could not find train_v2_drcat_02.csv. Place the file in this folder or set DATA_PATH to its full path.'
        )

df = pd.read_csv(DATA_PATH)

# ── Basic sanity checks ──────────────────────────────────────
print(f'Dataset shape  : {df.shape}')
print(f'Columns        : {list(df.columns)}')
print(f'\nLabel distribution (0=Human, 1=AI):')
print(df['label'].value_counts())
print(f'\nNull values:')
print(df[['text', 'label']].isnull().sum())
df.head(3)

## Cell 3 — Advanced Text Preprocessing

Each essay is passed through a 5-step cleaning pipeline:
1. Lowercase
2. Strip HTML tags
3. Remove non-alphabetic characters
4. Remove stopwords
5. Lemmatize tokens

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """
    Full preprocessing pipeline.
    Returns a cleaned, lemmatized string ready for TF-IDF vectorisation.
    """
    # Step 1: Lowercase — normalise vocabulary
    text = text.lower()

    # Step 2: Remove HTML tags (handles web-scraped content)
    text = re.sub(r'<[^>]+>', ' ', text)

    # Step 3: Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Step 4 & 5: Tokenise, remove stopwords, lemmatize
    tokens = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words and len(word) > 2
    ]

    return ' '.join(tokens)


# Drop any rows where 'text' is null before cleaning
df = df.dropna(subset=['text', 'label']).reset_index(drop=True)

print('Cleaning text — this may take ~60 seconds on the full dataset...')
df['clean_text'] = df['text'].astype(str).apply(clean_text)
print('Preprocessing complete.')

# Show a side-by-side sample
df[['text', 'clean_text', 'label']].head(3)

## Cell 4 — Train / Test Split

`stratify=y` ensures the class ratio is preserved in both splits, which is critical when the dataset may be imbalanced.

In [ ]:
X = df['clean_text']
y = df['label']          # 0 = Human, 1 = AI

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 80% train, 20% test
    random_state=42,
    stratify=y           # preserve class balance in both splits
)

print(f'Train size : {len(X_train):,}')
print(f'Test  size : {len(X_test):,}')
print(f'\nTrain label distribution:')
print(y_train.value_counts())
print(f'\nTest label distribution:')
print(y_test.value_counts())

## Cell 5 — TF-IDF Feature Engineering

Key settings:
- **`ngram_range=(1, 2)`** — captures both individual words and two-word phrases (bigrams), which are strong indicators of AI writing style
- **`max_features=10000`** — top 10,000 most informative n-grams
- **`sublinear_tf=True`** — applies log scaling to term frequencies

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # unigrams + bigrams
    max_features=10_000,    # vocabulary cap
    sublinear_tf=True,      # log(1 + tf) dampening
    min_df=2                # ignore very rare terms (appear < 2 docs)
)

# Fit ONLY on training data to prevent data leakage
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)      # transform only

print(f'Feature matrix shape (train) : {X_train_tfidf.shape}')
print(f'Feature matrix shape (test)  : {X_test_tfidf.shape}')
print(f'Actual vocabulary size       : {len(vectorizer.vocabulary_):,}')

## Cell 6 — Hyperparameter Tuning with RandomizedSearchCV

We search over 8 values of the regularisation parameter `C` using 5-fold cross-validation.  
A higher `C` = less regularisation (may overfit); lower `C` = more regularisation (may underfit).  
The search picks the value that maximises validation accuracy.

In [ ]:
# Base model with balanced class weights
base_model = LogisticRegression(
    class_weight='balanced',  # handles class imbalance automatically
    solver='saga',            # efficient for large sparse matrices
    max_iter=1000,            # ensure convergence
    random_state=42
)

# Search space for the regularisation parameter C
param_dist = {
    'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=8,           # test all 8 C values
    cv=5,               # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,          # use all available CPU cores
    random_state=42,
    verbose=1
)

print('Running hyperparameter search — 5-fold CV over 8 C values...')
search.fit(X_train_tfidf, y_train)

best_C  = search.best_params_['C']
best_cv = search.best_score_

print(f'\nBest C parameter   : {best_C}')
print(f'Best CV Accuracy   : {best_cv:.4f}  ({best_cv*100:.2f}%)')

# Show full CV results table
cv_results = pd.DataFrame(search.cv_results_)[['param_C', 'mean_test_score', 'std_test_score']]
cv_results = cv_results.sort_values('mean_test_score', ascending=False)
cv_results.columns = ['C', 'Mean CV Accuracy', 'Std Dev']
cv_results.round(4)

## Cell 7 — Train Final Model on Full Training Set

In [ ]:
# Train final model using the best C found by the search
model = LogisticRegression(
    C=best_C,
    class_weight='balanced',
    solver='saga',
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)
print(f'Final model trained with C={best_C}.')

## Cell 8 — Evaluation

Evaluate on the held-out test set and print the full classification report.

In [ ]:
# Generate predictions on the unseen test set
y_pred = model.predict(X_test_tfidf)

# Overall accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print('─' * 55)

# Per-class precision, recall, f1-score
print(classification_report(
    y_test, y_pred,
    target_names=['Human (0)', 'AI (1)']
))

## Cell 9 — Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Predicted Human', 'Predicted AI'],
    yticklabels=['Actual Human',    'Actual AI']
)
plt.title('Confusion Matrix — AI vs. Human Text Classifier', fontsize=14, pad=12)
plt.ylabel('True Label',      fontsize=11)
plt.xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

## Cell 10 — Top 20 Most Predictive Features

The logistic regression coefficient for each feature tells us how strongly that word or bigram pushes the prediction toward AI (positive) or Human (negative).

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef          = model.coef_[0]   # shape: (n_features,)

# Top 10 features pointing toward AI  (highest positive coefficients)
top_ai_idx    = np.argsort(coef)[-10:][::-1]
# Top 10 features pointing toward Human (most negative coefficients)
top_human_idx = np.argsort(coef)[:10]

features = (
    [(feature_names[i], coef[i], 'AI')    for i in top_ai_idx] +
    [(feature_names[i], coef[i], 'Human') for i in top_human_idx]
)

feat_df = pd.DataFrame(features, columns=['Feature', 'Coefficient', 'Class'])
feat_df = feat_df.sort_values('Coefficient', ascending=True)

# Blue bars = AI indicators | Orange bars = Human indicators
colors = ['#e07b54' if c == 'Human' else '#4c72b0' for c in feat_df['Class']]

plt.figure(figsize=(10, 7))
plt.barh(feat_df['Feature'], feat_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('TF-IDF Logistic Regression Coefficient', fontsize=11)
plt.title(
    'Top 20 Most Predictive Features\n(Blue = AI indicator  |  Orange = Human indicator)',
    fontsize=13
)
plt.tight_layout()
plt.savefig('top_features.png', dpi=150)
plt.show()
print('Saved: top_features.png')

## Cell 11 — Save Model Artifacts

In [ ]:
# Serialise both the vectorizer and the trained model for later use
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('logistic_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Artifacts saved:')
print('  tfidf_vectorizer.pkl')
print('  logistic_model.pkl')

## Cell 12 — Quick Inference Demo

Test the full pipeline on arbitrary text inputs.

In [ ]:
def predict_text(raw_text: str) -> dict:
    """
    Run the complete pipeline on a raw string.
    Returns a dict with the predicted label, confidence, and raw probabilities.
    """
    cleaned = clean_text(raw_text)
    vec     = vectorizer.transform([cleaned])
    pred    = model.predict(vec)[0]
    proba   = model.predict_proba(vec)[0]
    label   = 'AI-Generated' if pred == 1 else 'Human-Written'
    return {
        'label'      : label,
        'confidence' : f'{max(proba)*100:.1f}%',
        'raw_proba'  : {'Human': round(proba[0], 4), 'AI': round(proba[1], 4)}
    }


# ── Sample 1: Typical AI-generated text ─────────────────────
sample_ai = (
    "Artificial intelligence represents a transformative paradigm shift "
    "in modern technological infrastructure, facilitating unprecedented "
    "advancements across diverse sectors including healthcare, finance, "
    "and educational frameworks. It is important to note that these "
    "developments necessitate careful consideration of ethical implications."
)

# ── Sample 2: Typical student-written text ───────────────────
sample_human = (
    "I think AI is pretty cool but also kinda scary sometimes. "
    "Like what if it takes everyones jobs?? my teacher says we gotta "
    "learn how to use it or we'll be left behind lol. idk i just think "
    "people should be more careful about it."
)

print('=== Sample 1 (AI-style text) ===')
print(predict_text(sample_ai))

print('\n=== Sample 2 (Human-style text) ===')
print(predict_text(sample_human))

print('\n=== Try your own text ===')
# Replace the string below and re-run this cell
custom_text = "Paste your own essay or paragraph here to test the classifier."
print(predict_text(custom_text))